In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('house_prices.csv')
data.head()

,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [3]:
data.shape


(187531, 21)

In [4]:
missing_sum=data.isnull().sum()
missing_sum=missing_sum[missing_sum>0]
missing_percentage=missing_sum/len(data)*100
missing_values =pd.DataFrame()
missing_values['missing_sum']=missing_sum
missing_values['missing_percentage']=missing_percentage
missing_values=missing_values.sort_values(by='missing_sum', ascending=False)
missing_values


,missing_sum,missing_percentage
Plot Area,187531,100.000000
Dimensions,187531,100.000000
Society,109678,58.485264
Super Area,107685,57.422506
Car Parking,103357,55.114621
overlooking,81436,43.425354
Carpet Area,80673,43.018488
facing,70233,37.451408
Ownership,65517,34.936624
Balcony,48935,26.094352


Data cleaning

In [5]:
data = data.drop(columns=['Dimensions', 'Plot Area','Society','Super Area','Car Parking'])

In [6]:
data['overlooking'] = data['overlooking'].fillna('Unknown')

In [7]:
data['Carpet Area'] 

0          500 sqft
1          473 sqft
2          779 sqft
3          530 sqft
4          635 sqft
            ...    
187526          NaN
187527          NaN
187528    1250 sqft
187529          NaN
187530          NaN
Name: Carpet Area, Length: 187531, dtype: object

In [8]:
import re

def clean_area(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    
    # Lấy ra phần số trong chuỗi
    match = re.match(r'([0-9.]+)\s*(.*)', x)
    if match:
        val = float(match.group(1))
        unit = match.group(2).strip()
        
        # Quy đổi đơn vị về cùng sqft để model học chuẩn hơn
        if 'sqm' in unit or 'sq.m.' in unit:
            return val * 10.7639
        elif 'sqyrd' in unit:
            return val * 9.0
        
        return val # Mặc định là sqft
    return np.nan

In [9]:
data['Carpet Area'] = data['Carpet Area'].apply(clean_area)
data['Carpet Area'] = data['Carpet Area'].fillna(data['Carpet Area'].median())

In [10]:
data['Carpet Area'].isnull().sum()

np.int64(0)

In [13]:
data['facing'].value_counts()

facing
East            54741
North - East    24220
North           16533
West             8574
South            4694
North - West     3843
South - East     2622
South -West      2071
Name: count, dtype: int64

In [14]:
data['facing'] = data['facing'].fillna('Unknown')
data['facing'].value_counts()

facing
Unknown         70233
East            54741
North - East    24220
North           16533
West             8574
South            4694
North - West     3843
South - East     2622
South -West      2071
Name: count, dtype: int64

In [15]:
data['Ownership'].value_counts()

Ownership
Freehold                112229
Leasehold                 5285
Co-operative Society      3431
Power Of Attorney         1069
Name: count, dtype: int64

In [16]:
data['Ownership'].fillna('Unknown')

0                      Unknown
1                     Freehold
2                     Freehold
3                      Unknown
4         Co-operative Society
                  ...         
187526                Freehold
187527                 Unknown
187528                Freehold
187529                 Unknown
187530                Freehold
Name: Ownership, Length: 187531, dtype: object